In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c ai09-level1-project
!unzip ai09-level1-project.zip

!pip install ultralytics wandb scikit-learn

In [5]:
import os
import json
import glob
import shutil
import re
import yaml
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from PIL import Image

import wandb
from ultralytics import YOLO
from ultralytics import settings

# ==========================================
# 1. 경로 자동 탐색 및 설정
# ==========================================
possible_ann_dirs = glob.glob('/content/**/train_annotations', recursive=True)

TRAIN_ANN_DIR = possible_ann_dirs[0]
BASE_DIR = os.path.dirname(TRAIN_ANN_DIR)
TRAIN_IMG_DIR = os.path.join(BASE_DIR, "train_images")

possible_test_dirs = glob.glob('/content/**/test_images', recursive=True)
TEST_IMG_DIR = possible_test_dirs[0] if possible_test_dirs else os.path.join(BASE_DIR, "test_images")

YOLO_DATASET_DIR = "/content/yolo_dataset"
YAML_PATH = os.path.join(YOLO_DATASET_DIR, "dataset.yaml")

CLASS_MAPPING = {}
INV_CLASS_MAPPING = {}

# ==========================================
# 2. JSON 병합 및 YOLO 포맷 전처리
# ==========================================
def preprocess_data():
    print("\n1. 데이터 전처리")
    global CLASS_MAPPING, INV_CLASS_MAPPING
    CLASS_MAPPING.clear()
    INV_CLASS_MAPPING.clear()

    all_img_paths = glob.glob(os.path.join(TRAIN_IMG_DIR, '**', '*.[jp][pn]g'), recursive=True)
    img_path_map = {os.path.basename(p): p for p in all_img_paths}

    image_annotations = {}
    json_files = glob.glob(os.path.join(TRAIN_ANN_DIR, '**', '*.json'), recursive=True)

    for json_path in tqdm(json_files, desc="JSON 파싱 중"):
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        images_data = data.get('images', {})
        img_info = images_data[0] if isinstance(images_data, list) and len(images_data) > 0 else images_data

        if not isinstance(img_info, dict):
            continue

        file_name = img_info.get('file_name')
        img_width = img_info.get('width')
        img_height = img_info.get('height')
        dl_idx_str = img_info.get('dl_idx')

        if not file_name or dl_idx_str is None:
            continue

        category_id_raw = int(dl_idx_str)

        if category_id_raw not in CLASS_MAPPING:
            mapped_id = len(CLASS_MAPPING)
            CLASS_MAPPING[category_id_raw] = mapped_id
            INV_CLASS_MAPPING[mapped_id] = category_id_raw

        category_id_yolo = CLASS_MAPPING[category_id_raw]

        anns = data.get('annotations', [])
        for ann in anns:
            bbox = ann.get('bbox')
            if not bbox or len(bbox)!= 4:
                continue

            x_min, y_min, w, h = bbox
            x_center = max(0.0, min(1.0, (x_min + w / 2.0) / img_width))
            y_center = max(0.0, min(1.0, (y_min + h / 2.0) / img_height))
            norm_w = max(0.0, min(1.0, w / img_width))
            norm_h = max(0.0, min(1.0, h / img_height))

            yolo_line = f"{category_id_yolo} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"

            if file_name not in image_annotations:
                image_annotations[file_name] = []
            image_annotations[file_name].append(yolo_line)

    valid_images = list(image_annotations.keys())

    train_imgs, val_imgs = train_test_split(valid_images, test_size=0.2, random_state=42)

    def create_split(split_name, img_list):
        img_out_dir = os.path.join(YOLO_DATASET_DIR, 'images', split_name)
        lbl_out_dir = os.path.join(YOLO_DATASET_DIR, 'labels', split_name)
        os.makedirs(img_out_dir, exist_ok=True)
        os.makedirs(lbl_out_dir, exist_ok=True)

        for file_name in tqdm(img_list, desc=f"{split_name} 세트 구성 중"):
            src_img_path = img_path_map.get(file_name)
            if not src_img_path or not os.path.exists(src_img_path):
                continue

            shutil.copy(src_img_path, os.path.join(img_out_dir, file_name))

            base_name = os.path.splitext(file_name)[0]
            txt_path = os.path.join(lbl_out_dir, f"{base_name}.txt")
            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write("\n".join(image_annotations[file_name]))

    create_split('train', train_imgs)
    create_split('val', val_imgs)


# ==========================================
# 3. dataset.yaml 동적 생성
# ==========================================
def create_yaml():
    print("\n2. dataset.yaml 파일 생성")
    yaml_data = {
        'path': YOLO_DATASET_DIR,
        'train': 'images/train',
        'val': 'images/val',
        'nc': len(CLASS_MAPPING),
        'names': {k: str(v) for k, v in INV_CLASS_MAPPING.items()}
    }
    with open(YAML_PATH, 'w', encoding='utf-8') as f:
        yaml.dump(yaml_data, f, sort_keys=False)


# ==========================================
# 4. W&B 연동 및 모델 학습
# ==========================================
def train_model():
    print("\n3. W&B 연동 및 모델 학습")

    settings.update({"wandb": True})

    wandb.login()
    os.environ["WANDB_ENTITY"] = "team5pj1"
    os.environ["WANDB_PROJECT"] = "health-eat-team5"

    model = YOLO("yolo11n.pt")

    results = model.train(
        data=YAML_PATH,
        epochs=30,
        imgsz=640,
        batch=16,
        patience=10,
        project="health-eat-team5",
        name="yolov11_colab_baseline",
        exist_ok=True
    )

    wandb.finish()

    best_pt_path = "/content/runs/detect/health-eat-team5/yolov11_colab_baseline/weights/best.pt"
    return best_pt_path


# ==========================================
# 5. 추론 및 Kaggle 파일 생성
# ==========================================
def run_inference(best_model_path):
    print("\n4. 테스트 이미지 추론 및 제출 파일 생성")
    model = YOLO(best_model_path)
    submission_rows = []
    annotation_id_counter = 1

    test_files = glob.glob(os.path.join(TEST_IMG_DIR, '**', '*.[jp][pn]g'), recursive=True)

    for img_path in tqdm(test_files, desc="추론 중"):
        filename = os.path.basename(img_path)
        num_matches = re.findall(r'\d+', filename)
        image_id = int("".join(num_matches)) if num_matches else 0

        results = model.predict(source=img_path, conf=0.1, imgsz=640, verbose=False)
        result = results[0] if isinstance(results, list) else results

        if len(result.boxes) == 0:
            continue

        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            pred_yolo_cls = int(box.cls.item())

            original_category_id = INV_CLASS_MAPPING.get(pred_yolo_cls, pred_yolo_cls)

            row = {
                "annotation_id": annotation_id_counter,
                "image_id": image_id,
                "category_id": original_category_id,
                "bbox_x": x1,
                "bbox_y": y1,
                "bbox_w": x2 - x1,
                "bbox_h": y2 - y1,
                "score": round(float(box.conf.item()), 5)
            }
            submission_rows.append(row)
            annotation_id_counter += 1

    df = pd.DataFrame(submission_rows)
    if not df.empty:
        df = df[["annotation_id", "image_id", "category_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"]]

    output_csv = "/content/team5_submission.csv"
    df.to_csv(output_csv, index=False)

# ==========================================
# 실행부
# ==========================================
if __name__ == "__main__":
    if os.path.exists(YOLO_DATASET_DIR):
        shutil.rmtree(YOLO_DATASET_DIR)

    preprocess_data()
    create_yaml()
    best_weight = train_model()
    run_inference(best_weight)



1. 데이터 전처리


val 세트 구성 중: 100%|██████████| 47/47 [00:00<00:00, 153.37it/s]
wandb: WARNING Calling wandb.login() after wandb.init() has no effect.



2. dataset.yaml 파일 생성

3. W&B 연동 및 모델 학습
Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov11_colab_baseline, nbs=64, nms=False, opset=None, optimize=Fals

wandb: WARNING Tried to log to step 1 that is less than the current step 3. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


       2/30      2.95G     0.6837      4.536     0.9828         59        640: 100% ━━━━━━━━━━━━ 12/12 2.8it/s 4.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 6.5it/s 0.3s
                   all         47        155          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      2.95G     0.5788      4.453     0.9326        104        640: 33% ━━━━──────── 4/12 1.5it/s 3.0s<5.3s

wandb: WARNING Tried to log to step 2 that is less than the current step 3. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


       3/30      2.95G     0.5762      4.395     0.9307         67        640: 100% ━━━━━━━━━━━━ 12/12 2.0it/s 5.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 4.6it/s 0.4s
                   all         47        155          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/30      2.95G     0.5252      4.201     0.9103         54        640: 100% ━━━━━━━━━━━━ 12/12 2.8it/s 4.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 6.6it/s 0.3s
                   all         47        155          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       5/30      2.96G      0.506      4.019     0.9089         56        640: 100% ━━━━━━━━━━━━ 12/12 1.8it/s 6.7s
                 Class     Images  Instances      Box(P          R

lr/pg0,▂▃▄▅▆▇▇███▇▇▇▆▆▆▅▅▅▄▄▄▃▃▃▂▂▂▁▁
lr/pg1,▂▃▄▅▆▇▇███▇▇▇▆▆▆▅▅▅▄▄▄▃▃▃▂▂▂▁▁
lr/pg2,▂▃▄▅▆▇▇███▇▇▇▆▆▆▅▅▅▄▄▄▃▃▃▂▂▂▁▁
metrics/mAP50(B),▁▁▁▁▁▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇▇████
metrics/mAP50-95(B),▁▁▁▁▁▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇▇████
metrics/precision(B),▁▁▁▁▁▁▂▁▁▁▁▆▆▇▆▆▆▆▆▅▅██▇▇▇▇▇▆▇
metrics/recall(B),▁▁▁▁▁▂▄▅▇▇█▃▃▃▃▄▄▄▄▅▅▃▃▄▄▅▅▅▅▅
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...



4. 테스트 이미지 추론 및 제출 파일 생성


추론 중: 100%|██████████| 842/842 [00:46<00:00, 18.26it/s]
